## Preparación del entorno

Localizamos la carpeta del proyecto e importamos las librerías
y funciones necesarias para ejecutar la auditoría.

In [1]:
from pathlib import Path
import os

# Localizamos la raíz para acceder al código y a los datos.
raiz_proyecto = Path.cwd().resolve()

if raiz_proyecto.name == "notebooks":
    raiz_proyecto = raiz_proyecto.parent

if not (raiz_proyecto / "main.py").is_file():
    raise FileNotFoundError(
        "Abre Jupyter desde la carpeta del proyecto."
    )

os.chdir(raiz_proyecto)

In [2]:
# Librerías y funciones utilizadas en el notebook.
import sys

import numpy as np
import pandas as pd

from main import ejecutar_pipeline
from src.ingesta.codificacion import (
    codificar_binaria,
    codificar_one_hot,
    codificar_ordinal,
    dataframe_a_tuplas,
)
from src.ingesta.escalado import escalar_columnas_min_max
from src.ingesta.lectura import cargar_dataset
from src.ingesta.limpieza import imputar_mediana, pipe
from src.ingesta.outliers import (
    detectar_outliers_iqr,
    filtrar_outliers_iqr,
)
from src.modelo.distancias import calcular_matriz_distancias
from src.modelo.knn import predecir_knn
from src.modelo.transformacion import (
    tuplas_a_matriz,
    tuplas_a_vector,
)

pd.set_option("display.max_columns", None)

In [3]:
# Cargamos los datos originales.
datos_originales = cargar_dataset("data/raw/credit_risk_dataset.csv")

print(f"Filas: {datos_originales.shape[0]}")
print(f"Columnas: {datos_originales.shape[1]}")

display(datos_originales.head())


Filas: 32581
Columnas: 12


,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length
0,22,59000,RENT,123.0,PERSONAL,D,35000,16.02,1,0.59,Y,3
1,21,9600,OWN,5.0,EDUCATION,B,1000,11.14,0,0.10,N,2
2,25,9600,MORTGAGE,1.0,MEDICAL,C,5500,12.87,1,0.57,N,3
3,23,65500,RENT,4.0,MEDICAL,C,35000,15.23,1,0.53,N,2
4,24,54400,RENT,8.0,MEDICAL,C,35000,14.27,1,0.55,Y,4


In [4]:
# Contamos valores nulos sin modificar los datos.
valores_nulos = datos_originales.isnull().sum()

print("Valores nulos por columna:")
print(valores_nulos)

Valores nulos por columna:
person_age                       0
person_income                    0
person_home_ownership            0
person_emp_length              895
loan_intent                      0
loan_grade                       0
loan_amnt                        0
loan_int_rate                 3116
loan_status                      0
loan_percent_income              0
cb_person_default_on_file        0
cb_person_cred_hist_length       0
dtype: int64


In [5]:
# Demostramos la composición de funciones para imputar nulos.
# La evaluación final tendrá su propio preprocesamiento en main.py.
datos_imputados = pipe(
    datos_originales,
    lambda datos: imputar_mediana(
        datos,
        "person_emp_length",
    ),
    lambda datos: imputar_mediana(
        datos,
        "loan_int_rate",
    ),
)

columnas_nulos = [
    "person_emp_length",
    "loan_int_rate",
]

print("Nulos antes:")
print(datos_originales[columnas_nulos].isnull().sum())

print("\nNulos después:")
print(datos_imputados[columnas_nulos].isnull().sum())

Nulos antes:
person_emp_length     895
loan_int_rate        3116
dtype: int64

Nulos después:
person_emp_length    0
loan_int_rate        0
dtype: int64


In [6]:
# Detectamos y filtramos outliers de años de empleo.
outliers_empleo = detectar_outliers_iqr(
    datos_imputados,
    "person_emp_length",
)

datos_sin_outliers = filtrar_outliers_iqr(
    datos_imputados,
    "person_emp_length",
)

print(f"Filas antes:\t{len(datos_imputados)}")
print(f"Outliers detectados:\t{len(outliers_empleo)}")
print(f"Filas después:\t{len(datos_sin_outliers)}")

print(
    "Máximo antes:",
    datos_imputados["person_emp_length"].max(),
)
print(
    "Máximo después:",
    datos_sin_outliers["person_emp_length"].max(),
)

Filas antes:	32581
Outliers detectados:	853
Filas después:	31728
Máximo antes: 123.0
Máximo después: 14.0


In [7]:
# Demostramos el escalado Min-Max con map, filter y reduce.
columnas_a_escalar = [
    "person_age",
    "person_income",
    "person_emp_length",
    "loan_amnt",
    "loan_int_rate",
    "loan_percent_income",
    "cb_person_cred_hist_length",
]

datos_escalados = escalar_columnas_min_max(
    datos_sin_outliers,
    columnas_a_escalar,
)

print("Valores mínimos después del escalado:")
print(datos_escalados[columnas_a_escalar].min())

print("\nValores máximos después del escalado:")
print(datos_escalados[columnas_a_escalar].max())

Valores mínimos después del escalado:
person_age                    0.0
person_income                 0.0
person_emp_length             0.0
loan_amnt                     0.0
loan_int_rate                 0.0
loan_percent_income           0.0
cb_person_cred_hist_length    0.0
dtype: float64

Valores máximos después del escalado:
person_age                    1.0
person_income                 1.0
person_emp_length             1.0
loan_amnt                     1.0
loan_int_rate                 1.0
loan_percent_income           1.0
cb_person_cred_hist_length    1.0
dtype: float64


In [8]:
# Comparamos el original en memoria con el CSV sin modificar.
datos_recargados = cargar_dataset("data/raw/credit_risk_dataset.csv")

print(
    "El dataset original no fue modificado:",
    datos_originales.equals(datos_recargados),
)

El dataset original no fue modificado: True


# Motor de computo vectorizado y clasificador KNN desde cero
KNN utiliza unicamente NumPy.


In [9]:
# Codificamos categorías como números para calcular distancias.
datos_numericos = pipe(
    datos_escalados,
    lambda datos: codificar_binaria(
        datos,
        "cb_person_default_on_file",
        "Y",
    ),
    lambda datos: codificar_ordinal(
        datos,
        "loan_grade",
        list("ABCDEFG"),
    ),
    lambda datos: codificar_one_hot(
        datos,
        "person_home_ownership",
    ),
    lambda datos: codificar_one_hot(
        datos,
        "loan_intent",
    ),
)

print(f"Columnas antes:\t{datos_escalados.shape[1]}")
print(f"Columnas después:\t{datos_numericos.shape[1]}")
print(f"Nulos después:\t{datos_numericos.isnull().sum().sum()}")

Columnas antes:	12
Columnas después:	20
Nulos después:	0


In [10]:
# Separamos características y etiquetas.
caracteristicas = datos_numericos.drop(
    columns=["loan_status"]
)
respuesta = datos_numericos["loan_status"]

filas_x = dataframe_a_tuplas(caracteristicas)
valores_y = tuple(respuesta)

matriz_x = tuplas_a_matriz(filas_x)
vector_y = tuplas_a_vector(valores_y)

print(f"Forma de X: {matriz_x.shape}")
print(f"Forma de y: {vector_y.shape}")
print(f"Se puede modificar X: {matriz_x.flags.writeable}")
print(f"Se puede modificar y: {vector_y.flags.writeable}")

Forma de X: (31728, 19)
Forma de y: (31728,)
Se puede modificar X: False
Se puede modificar y: False


In [11]:
# Prueba reducida para demostrar el funcionamiento del KNN.
# No corresponde a la evaluación final del modelo.
np.random.seed(42)

posiciones = np.random.permutation(len(matriz_x))
corte = int(0.8 * len(matriz_x))

posiciones_train = posiciones[:corte][:1000]
posiciones_test = posiciones[corte:][:200]

x_train = matriz_x[posiciones_train]
y_train = vector_y[posiciones_train]

x_test = matriz_x[posiciones_test]
y_test = vector_y[posiciones_test]

print(f"Xtrain: {x_train.shape}  (M x D)")
print(f"Xtest: {x_test.shape}  (N x D)")

Xtrain: (1000, 19)  (M x D)
Xtest: (200, 19)  (N x D)


In [12]:
# Calculamos distancias sin bucles.
distancias = calcular_matriz_distancias(
    x_train,
    x_test,
)

print(
    "Forma de la matriz de distancias:",
    distancias.shape,
    "(N x M)",
)

# Comparamos un bloque con la fórmula directa.
directa = np.sqrt(
    (
        (x_test[:5, None, :] - x_train[None, :50, :]) ** 2
    ).sum(axis=2)
)

print(
    "Coincide con la fórmula directa:",
    np.allclose(distancias[:5, :50], directa),
)

Forma de la matriz de distancias: (200, 1000) (N x M)
Coincide con la fórmula directa: True


In [13]:
# Demostramos la predicción por votación mayoritaria.
predicciones = predecir_knn(
    x_train,
    y_train,
    x_test,
    k=5,
)

exactitud = np.mean(predicciones == y_test)

print(f"Predicciones: {predicciones.shape[0]}")
print(f"Clase 0 predicha: {np.sum(predicciones == 0)}")
print(f"Clase 1 predicha: {np.sum(predicciones == 1)}")
print(f"Exactitud preliminar: {exactitud:.3f}")

Predicciones: 200
Clase 0 predicha: 170
Clase 1 predicha: 30
Exactitud preliminar: 0.835


## Reproducibilidad, métricas y auditoría de sesgo

Esta sección se conecta con `main.py`, que reutiliza el preprocesamiento del Hito 1 y el clasificador KNN del Hito 2. Se fija una semilla para reproducir la partición 80/20 y se evalúan las predicciones con métricas globales y por grupo de edad.

**Nota:** `loan_status=0` se usa solo como proxy exploratorio de resultado favorable; no significa que el banco haya aprobado realmente un crédito.


In [14]:
# Ejecutamos la evaluación final del proyecto.
# main.py separa los datos antes del preprocesamiento.
resultados = ejecutar_pipeline(mostrar=False)

print("Filas originales:", resultados["filas_originales"])
print(
    "Entrenamiento inicial:",
    resultados["filas_entrenamiento_inicial"],
)
print(
    "Outliers retirados del entrenamiento:",
    resultados["filas_outliers_retiradas"],
)
print(
    "Entrenamiento después de IQR:",
    resultados["filas_entrenamiento"],
)
print("Filas de prueba:", resultados["filas_prueba"])
print("Semilla:", resultados["semilla"])
print("Número de vecinos (k):", resultados["k"])


Filas originales: 32581
Entrenamiento inicial: 26064
Outliers retirados del entrenamiento: 686
Entrenamiento después de IQR: 25378
Filas de prueba: 6517
Semilla: 42
Número de vecinos (k): 5


In [15]:
# Mostramos las métricas de la evaluación final.
metricas_globales = pd.DataFrame(
    [resultados["metricas_globales"]]
).T.rename(columns={0: "valor"})

display(metricas_globales.round(4))

,valor
TP,810.0000
FP,128.0000
TN,4944.0000
FN,635.0000
precision,0.8635
recall,0.5606
f1_score,0.6798
accuracy,0.8829


In [16]:
# Calculamos el indicador exploratorio entre grupos.
disparate_impact_resultado = resultados["disparate_impact"]

display(
    pd.DataFrame([disparate_impact_resultado])
    .T.rename(columns={0: "valor"})
)

ratio = disparate_impact_resultado["disparate_impact"]

if pd.isna(ratio):
    print(
        "No se puede interpretar el ratio: "
        "falta una tasa válida."
    )
elif ratio < 0.80:
    print(
        "Señal exploratoria: investigar diferencias "
        "entre grupos y sus posibles causas."
    )
else:
    print(
        "El ratio no está por debajo de la referencia de 0.80; "
        "aún deben revisarse otras métricas y el contexto."
    )

,valor
grupo_protegido,menor_25
grupo_control,25_o_mas
tasa_favorable_protegido,0.848397
tasa_favorable_control,0.860544
disparate_impact,0.985884


El ratio no está por debajo de la referencia de 0.80; aún deben revisarse otras métricas y el contexto.


### Prevención de fuga de datos y reproducibilidad

Se utiliza la semilla 42 para repetir la partición aleatoria.
El dataset original se divide en entrenamiento y prueba antes
de realizar el preprocesamiento.

Las medianas y los parámetros Min-Max se calculan exclusivamente
con entrenamiento y se reutilizan en prueba. El filtrado IQR
afecta únicamente al entrenamiento.

Este orden evita que los datos de prueba intervengan en el ajuste
de la imputación y del escalado.

In [17]:
# Identificamos el intérprete que utiliza el notebook.
print("Versión de Python:", sys.version.split()[0])
print("Python utilizado:", sys.executable)
print(
    "Entorno virtual activo:",
    sys.prefix != sys.base_prefix,
)

Versión de Python: 3.13.9
Python utilizado: C:\Users\ASUS\anaconda3\python.exe
Entorno virtual activo: False


### Conclusión de auditoría

El clasificador se evaluó con una partición inicial reproducible de
80 % para entrenamiento y 20 % para prueba, utilizando la semilla 42.
Las medianas y los parámetros de escalado se calcularon únicamente
con entrenamiento, evitando utilizar información de prueba.

Los resultados muestran que la exactitud global debe interpretarse
junto con Precision, Recall y F1. En la ejecución verificada, la
exactitud fue de 88,29 %, pero el Recall de la clase 1 fue de 56,06 %.
El modelo dejó sin detectar 635 de los 1445 casos reales de esa clase.

La auditoría comparó menores de 25 años con personas de 25 años o más.
Las tasas de predicción favorable fueron de 84,84 % y 86,05 %,
respectivamente, con un Disparate Impact de aproximadamente 0,9859.
Estas tasas son cercanas, aunque existen diferencias en las métricas
de clasificación entre los grupos.

### Estrategia de mitigación propuesta

Se propone revisar la representación de cada grupo y de las clases
0 y 1 dentro del entrenamiento. Si se identifica una representación
insuficiente, se evaluará un muestreo más equilibrado únicamente
sobre los datos de entrenamiento.

También se propone comparar distintos valores impares de k mediante
una división de validación dentro del entrenamiento. La selección
considerará el Recall y F1 de la clase 1, las métricas por edad y las
diferencias entre tasas favorables.

La prueba final se mantendrá separada durante estos ajustes. Una vez
seleccionada la alternativa, se compararán sus resultados con los del
modelo inicial para comprobar si mejora la detección de la clase 1
sin ampliar las diferencias entre grupos.

Estas medidas son propuestas; todavía no se han aplicado ni se ha
demostrado que mejoren el modelo.

### Limitaciones

La predicción loan_status=0 se utiliza como indicador exploratorio de
resultado favorable y no representa una aprobación bancaria real.
Un Disparate Impact cercano a uno no demuestra ausencia de sesgo.

Excluir la edad del clasificador tampoco garantiza equidad, porque
otras variables pueden relacionarse con ella. Además, deben revisarse
los registros con edades extremas y justificarse los grupos elegidos.

Las conclusiones corresponden a este dataset y a esta partición.


In [18]:
from pathlib import Path

carpeta_modelo = Path(resultados["carpeta_modelo"])

print("Carpeta:", carpeta_modelo)
print("Modelo guardado:", (carpeta_modelo / "modelo.npz").is_file())
print(
    "Configuración guardada:",
    (carpeta_modelo / "configuracion.json").is_file(),
)

Carpeta: C:\PROYECTO\ENTREGA_PROYECTO_RIESGO_CREDITICIO_INTEGRADO - copia\proyecto_riesgo_crediticio\models\knn_20261016_135046_285085
Modelo guardado: True
Configuración guardada: True


In [19]:
from src.pipeline import RUTA_DATOS, SEMILLA, preparar_datos

# Reproducimos la partición utilizada en el pipeline.
datos_revision = cargar_dataset(str(RUTA_DATOS))

posiciones = np.random.RandomState(SEMILLA).permutation(
    len(datos_revision)
)
corte = int(0.8 * len(datos_revision))

train_original = datos_revision.iloc[posiciones[:corte]].copy()
test_original = datos_revision.iloc[posiciones[corte:]].copy()

train_preparado, test_preparado = preparar_datos(
    train_original,
    test_original,
)

# Recuperamos las filas originales conservadas después del IQR.
train_filtrado = train_original.loc[train_preparado.index]


def resumir_muestra(datos):
    """Resume clases y edades sin modificar los datos."""
    return pd.Series({
        "filas": len(datos),
        "porcentaje_clase_1": (
            datos["loan_status"].eq(1).mean() * 100
        ),
        "porcentaje_menores_25": (
            datos["person_age"].lt(25).mean() * 100
        ),
    })


comparacion = pd.DataFrame({
    "Dataset original": resumir_muestra(datos_revision),
    "Entrenamiento inicial": resumir_muestra(train_original),
    "Entrenamiento tras IQR": resumir_muestra(train_filtrado),
    "Prueba": resumir_muestra(test_original),
}).T

display(comparacion.round(2))

print(
    "Prueba conserva las filas y su orden:",
    test_preparado.index.equals(test_original.index),
)

,filas,porcentaje_clase_1,porcentaje_menores_25
Dataset original,32581.0,21.82,37.80
Entrenamiento inicial,26064.0,21.73,38.04
Entrenamiento tras IQR,25378.0,21.86,39.06
Prueba,6517.0,22.17,36.84


Prueba conserva las filas y su orden: True


### Revisión de la composición de las muestras

El filtrado IQR retiró 686 registros del entrenamiento.
La proporción de la clase 1 pasó de 21,73 % a 21,86 %,
mientras que la proporción de menores de 25 años pasó
de 38,04 % a 39,06 %.

El conjunto de prueba conservó sus 6517 registros y su orden.
Las medianas y los parámetros de escalado se calcularon
exclusivamente con entrenamiento.

Estos resultados permiten observar cambios en la composición,
pero no demuestran que el dataset represente a la población real
ni descartan sesgo de selección.